<a href="https://colab.research.google.com/github/kkorotkovuky/bch612-colab/blob/main/AlphaFold_single_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold2 – single sequence input (BCH612, 2026 update)
- For DEMO and educational purposes only.
- For natural proteins you usually need a multiple sequence alignment to get an accurate prediction (see [ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)). With a single sequence, AlphaFold succeeds mainly on idealized, *de novo*-like sequences, which is what makes it useful for learning the design principles of proteins.
- Exercise and original notebook (`af_backprop/AlphaFold_single.ipynb`) by Sergey Ovchinnikov. This version runs the same model (AlphaFold2 `model_3_ptm`, no MSA, no templates) through the maintained [ColabDesign](https://github.com/sokrypton/ColabDesign) code, so it works on the current Colab runtime (Python 3.13, JAX 0.11).

### Tips and instructions
- **Runtime → Change runtime type → T4 GPU** before you start.
- Click the ▶ play icon to the left of the cell below. The first run installs the software and downloads the AlphaFold2 weights (about 2 min). After that, each new sequence takes a few seconds.
- Edit the sequence and press ▶ again. Changing the length by more than 25 residues triggers a one-time recompile.
- Hydrophobic residues (A, G, V, I, L, F, M, P, W, C, Y) are drawn with yellow carbons, so you can see whether they form a core.
- Hover over an atom in the 3D view to see the residue name and number.
- Use `/` to separate chains (e.g. `AAAA/AAAA`).
- Check the **cyclic_peptide** box to fold the sequence as a head-to-tail macrocycle. This changes the relative-position encoding so that AlphaFold treats residue 1 and residue L as neighbors (the cyclic offset of Ovchinnikov, used in ColabDesign's `af_cyc_design`); no covalent bond is added to the force field, and D-amino acids and N-methylation, which real macrocycles rely on, cannot be represented. Compare the pLDDT and the printed CA(1)-CA(L) distance with and without the box checked.


In [ ]:
#@title Enter the amino acid sequence to fold ⬇️
sequence = 'GGGGGGGGGGGGGGGGGGGG' #@param {type:"string"}
recycles = 0 #@param ["0", "1", "2", "3", "6", "12", "24", "48"] {type:"raw"}
cyclic_peptide = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;*check the `cyclic_peptide` box to fold the chain as a head-to-tail macrocycle (N-terminus bonded to C-terminus)*
#@markdown #### Display options
color = "confidence" #@param ["confidence", "rainbow", "chain"]
show_sidechains = True #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}
PAE_colors = "AlphaFold (green-white)" #@param ["AlphaFold (green-white)", "ColabFold (red-white-blue)"]

###############################################################################
# SETUP (runs once per session)
###############################################################################
import os, sys, re, subprocess, warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

# ColabDesign pinned to the 14 Oct 2026 commit that fixed compatibility with new JAX
COLABDESIGN = "git+https://github.com/sokrypton/ColabDesign.git@8fd167d"
PARAMS_URL  = "https://storage.googleapis.com/alphafold/alphafold_params_2021-07-14.tar"
PARAMS_FILE = "params/params_model_3_ptm.npz"
PARAMS_SIZE = 371746284                       # bytes
PARAMS_BYTE_RANGE = "1861348352-2233094635"   # where params_model_3_ptm.npz sits inside the tar

def _sh(cmd):
  r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
  if r.returncode != 0:
    raise RuntimeError(f"Setup step failed:\n  {cmd}\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}")

if "SETUP_DONE" not in dir():
  try:
    import colabdesign
  except ImportError:
    print("Installing ColabDesign ...")
    _sh(f"{sys.executable} -m pip -q install {COLABDESIGN}")

  if not (os.path.isfile(PARAMS_FILE) and os.path.getsize(PARAMS_FILE) == PARAMS_SIZE):
    print("Downloading AlphaFold2 weights (model_3_ptm, 372 MB) ...")
    os.makedirs("params", exist_ok=True)
    try:   # fetch only the one file we need out of the 3.7 GB archive
      _sh(f"curl -fsS -r {PARAMS_BYTE_RANGE} -o {PARAMS_FILE} {PARAMS_URL}")
      assert os.path.getsize(PARAMS_FILE) == PARAMS_SIZE
    except Exception:
      print("  partial download failed; downloading the full archive instead (3.7 GB) ...")
      _sh(f"curl -fsSL {PARAMS_URL} | tar -x -C params ./params_model_3_ptm.npz")

  import numpy as np
  import jax, jax.numpy as jnp
  import matplotlib, matplotlib.pyplot as plt
  from matplotlib import animation
  if not hasattr(matplotlib.cm, "get_cmap"):   # removed in matplotlib 3.9; ColabDesign plotting still calls it
    matplotlib.cm.get_cmap = matplotlib.colormaps.get_cmap
  from IPython.display import HTML, display
  import tqdm.notebook
  TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

  from colabdesign.af.alphafold.model import config, data, model
  from colabdesign.af.alphafold.common import protein, residue_constants
  from colabdesign.af.prep import prep_input_features
  from colabdesign.af.inputs import update_seq, update_aatype
  from colabdesign.af.loss import get_plddt, get_pae
  from colabdesign.shared.plot import show_pdb, plot_pseudo_3D, pymol_cmap
  from colabdesign.shared.protein import _np_kabsch, renum_pdb_str
  from colabdesign.shared.utils import clear_mem

  DEVICE = jax.devices()[0].platform
  if DEVICE == "cpu":
    print("WARNING: no GPU detected. Use Runtime -> Change runtime type -> T4 GPU "
          "(CPU works, but is much slower).")
  else:
    print(f"Running on {DEVICE.upper()}")

  # AlphaFold2 model_3_ptm, single sequence, no templates, recycles run one at a time below
  cfg = config.model_config("model_3_ptm")
  cfg.model.num_recycle = 0
  cfg.model.global_config.use_remat = False
  cfg.model.global_config.use_dgram = False
  cfg.model.global_config.bfloat16 = True
  cfg.model.global_config.subbatch_size = None
  MODEL_PARAMS = data.get_model_haiku_params(model_name="model_3_ptm", data_dir=".", fuse=True)
  MODEL_PARAMS = {k: v for k, v in MODEL_PARAMS.items() if "template" not in k}
  RUNNER = model.RunModel(cfg, recycle_mode="last")

  def make_inputs(max_len):
    inputs = prep_input_features(L=max_len, N=1, T=1)
    zeros = np.zeros(max_len, dtype=int)
    inputs.update({"asym_id": zeros, "sym_id": zeros, "entity_id": zeros,
                   "use_dropout": False})
    return inputs

  def make_offset(residue_index, Ls, cyclic):
    """relative-position matrix AlphaFold uses to place residues in sequence space.
    For a macrocycle, residue 1 and residue L are made neighbors (Sergey Ovchinnikov's
    cyclic offset, as in ColabDesign's af_cyc_design)."""
    offset = residue_index[:, None] - residue_index[None, :]
    if cyclic:
      start = 0
      for L in Ls:                      # each chain is cyclised separately
        i = np.arange(L)
        ij = np.stack([i, i + L], -1)
        lin = i[:, None] - i[None, :]
        c = np.abs(ij[:, None, :, None] - ij[None, :, None, :]).min((2, 3))
        shorter = c < np.abs(lin)
        c[shorter] = -c[shorter]
        offset[start:start + L, start:start + L] = c * np.sign(lin)
        start += L
    return offset.astype(np.int32)

  @jax.jit
  def fold_once(model_params, inputs, aatype, length, residue_index, offset, prev):
    # one pass through AlphaFold2; positions >= length are padding and are masked out
    inputs = dict(inputs)
    inputs["offset"] = offset
    mask = (jnp.arange(aatype.shape[0]) < length).astype(jnp.float32)
    inputs["seq_mask"] = mask
    inputs["msa_mask"] = jnp.broadcast_to(mask, inputs["msa_mask"].shape)
    inputs["residue_index"] = residue_index
    aa = jnp.where(mask > 0, aatype, 0)
    update_seq(jax.nn.one_hot(aa, 20)[None] * mask[None, :, None], inputs)
    update_aatype(aa, inputs)
    inputs["prev"] = prev
    out = RUNNER.apply(model_params, jax.random.PRNGKey(0), inputs)
    return {"xyz":       out["structure_module"]["final_atom_positions"],
            "atom_mask": out["structure_module"]["final_atom_mask"],
            "plddt":     get_plddt(out),
            "pae":       get_pae(out),
            "prev":      out["prev"]}

  def save_pdb(o, aatype, Ls, filename, cyclic=False):
    L = sum(Ls)
    p = protein.Protein(atom_positions=o["xyz"][:L], atom_mask=o["atom_mask"][:L],
                        aatype=aatype[:L], residue_index=np.arange(L),
                        b_factors=100 * o["plddt"][:L, None] * o["atom_mask"][:L])
    pdb_str = "\n".join(protein.to_pdb(p).splitlines()[1:-2])
    pdb_str = renum_pdb_str(pdb_str, Ls)

    # disulfide bonds: SG atoms closer than 3 A, the same test OpenMM uses when
    # it builds a topology, written as standard SSBOND records so that viewers
    # draw them and the MD notebook can report them
    def chain_and_resnum(i):
      start = 0
      for n, Lc in enumerate(Ls):
        if i < start + Lc: return "ABCDEFGH"[n % 8], i - start + 1
        start += Lc
    sg_idx = residue_constants.atom_order["SG"]
    cys = [i for i in range(L) if residue_constants.restypes[aatype[i]] == "C"
           and o["atom_mask"][i, sg_idx] > 0]
    ss_pairs, ss_lines = [], []
    for a in range(len(cys)):
      for b in range(a + 1, len(cys)):
        i, j = cys[a], cys[b]
        d = float(np.linalg.norm(o["xyz"][i, sg_idx] - o["xyz"][j, sg_idx]))
        if d < 3.0:
          c1, r1 = chain_and_resnum(i); c2, r2 = chain_and_resnum(j)
          ss_pairs.append((c1, r1, c2, r2, d))
    for n, (c1, r1, c2, r2, d) in enumerate(ss_pairs, 1):
      row = [" "] * 80
      def put(col, text, row=row): row[col - 1:col - 1 + len(text)] = list(text)
      put(1, "SSBOND"); put(8, f"{n:>3d}"); put(12, "CYS"); put(16, c1); put(18, f"{r1:>4d}")
      put(26, "CYS"); put(30, c2); put(32, f"{r2:>4d}")
      put(62, "1555"); put(69, "1555"); put(74, f"{min(d, 99.99):>5.2f}")
      ss_lines.append("".join(row).rstrip())
    if cyclic:
      # LINK record for the head-to-tail bond, in the column layout used by deposited
      # cyclic peptides (e.g. 1NB1 kalata B1, 1SFI SFTI-1), so viewers draw the ring closed
      def link_record(first3, last3, Lc, ch):
        row = [" "] * 80
        def put(col, text): row[col - 1:col - 1 + len(text)] = list(text)
        put(1, "LINK  ")
        put(14, "N"); put(18, f"{first3:>3s}"); put(22, ch); put(23, f"{1:>4d}")
        put(44, "C"); put(48, f"{last3:>3s}");  put(52, ch); put(53, f"{Lc:>4d}")
        put(62, "1555"); put(69, "1555"); put(74, f"{1.33:>5.2f}")
        return "".join(row).rstrip()
      links, start = [], 0
      r1, r3 = residue_constants.restype_1to3, residue_constants.restypes
      for n, Lc in enumerate(Ls):
        ch = "ABCDEFGH"[n % 8]
        links.append(link_record(r1[r3[aatype[start]]], r1[r3[aatype[start + Lc - 1]]], Lc, ch))
        start += Lc
      ss_lines = ss_lines + links   # PDB record order: SSBOND, then LINK
    if ss_lines:
      pdb_str = "\n".join(ss_lines) + "\n" + pdb_str
    pdb_str += "\nEND\n"
    with open(filename, "w") as f: f.write(pdb_str)
    return ss_pairs

  def plot_confidence(plddt, pae, Ls=None, dpi=100, style="AlphaFold (green-white)"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5), dpi=dpi)
    ax1.plot(plddt, color="black"); ax1.set_ylim(0, 100)
    ax1.set_title("Predicted lDDT"); ax1.set_xlabel("position"); ax1.set_ylabel("pLDDT")
    # AlphaFold (AlphaFold DB, AlphaFold Server): dark green = low error, white = high, 0-31.75 A
    # ColabFold: blue = low error, white = middle, red = high, 0-30 A
    if style.startswith("ColabFold"):
      im = ax2.imshow(pae, cmap="bwr", vmin=0, vmax=30)
    else:
      im = ax2.imshow(pae, cmap="Greens_r", vmin=0, vmax=31.75)
    # PAE(i, j) = expected error at residue j when the model is aligned on residue i; rows (y) are i
    ax2.set_title("Predicted aligned error"); ax2.set_xlabel("scored residue"); ax2.set_ylabel("aligned residue")
    fig.colorbar(im, ax=ax2, label="PAE (Å)", fraction=0.046, pad=0.04)
    if Ls is not None and len(Ls) > 1:
      for b in np.cumsum(Ls)[:-1]:
        ax1.axvline(b - 0.5, color="gray", lw=0.8)
        ax2.axvline(b - 0.5, color="black", lw=0.8); ax2.axhline(b - 0.5, color="black", lw=0.8)
    fig.tight_layout()
    return fig

  def make_animation(positions, plddts, Ls=None, line_w=2.0, dpi=100):
    # 2D pseudo-3D movie of the CA trace across recycles, aligned to the last recycle
    ca = [p[:, 1, :] - p[:, 1, :].mean(0, keepdims=True) for p in positions]
    view = ca[-1] @ np.asarray(_np_kabsch(ca[-1], ca[-1], return_v=True, use_jax=False))
    pos = np.asarray([c @ np.asarray(_np_kabsch(c, view, use_jax=False)) for c in ca])
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 5), dpi=dpi)
    fig.subplots_adjust(top=0.90, bottom=0.10, right=1, left=0, hspace=0, wspace=0)
    lo, hi = pos[..., :2].min() - 1, pos[..., :2].max() + 1
    for ax in [ax1, ax3]:
      ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.axis(False)
    ax2.set_xlabel("positions"); ax2.set_ylabel("pLDDT"); ax2.set_ylim(0, 100)
    def txt(t, ax):
      return ax.text(0.5, 1.01, t, horizontalalignment="center",
                     verticalalignment="bottom", transform=ax.transAxes)
    frames = []
    for k, (xyz, plddt) in enumerate(zip(pos, plddts)):
      f = [ax2.plot(plddt, animated=True, color="black")[0],
           txt(f"recycle={k}", ax2), txt(f"pLDDT={plddt.mean():.1f}", ax3)]
      if Ls is None or len(Ls) == 1:
        f += [txt("colored by N->C", ax1), plot_pseudo_3D(xyz, ax=ax1, line_w=line_w)]
      else:
        c = np.concatenate([[n] * L for n, L in enumerate(Ls)])
        f += [txt("colored by chain", ax1),
              plot_pseudo_3D(xyz, c=c, Ls=Ls, cmap=pymol_cmap, cmin=0, cmax=39, line_w=line_w, ax=ax1)]
      f += [plot_pseudo_3D(xyz, c=plddt, Ls=Ls, cmin=50, cmax=90, line_w=line_w, ax=ax3)]
      frames.append(f)
    ani = animation.ArtistAnimation(fig, frames, blit=True, interval=120)
    plt.close()
    return ani.to_html5_video()

  current_seq, max_length = "", -1
  SETUP_DONE = True

###############################################################################
# PREDICTION
###############################################################################
ori_sequence = re.sub(r"[^A-Z/:]", "", sequence.upper()).strip("/:")
Ls = [len(s) for s in re.split(r"[/:]", ori_sequence) if len(s) > 0]
seq_str = re.sub("[^A-Z]", "", ori_sequence)
length = len(seq_str)
assert length > 0, "Please enter an amino acid sequence."
recycles = int(recycles)

# avoid recompiling if the length changes by 25 residues or less
if length > max_length or (max_length - length) > 25:
  max_length = length + 25
  INPUTS = make_inputs(max_length)
  current_seq = ""

if (ori_sequence, cyclic_peptide) != current_seq:
  aatype = np.array([residue_constants.restype_order.get(a, 0) for a in seq_str])
  aatype = np.pad(aatype, [0, max_length - length])
  # residue numbering, with a gap of 50 between chains (chain breaks)
  res_idx = np.concatenate([np.arange(L) + 50 * n + sum(Ls[:n]) for n, L in enumerate(Ls)])
  res_idx = np.concatenate([res_idx, res_idx[-1] + 1 + np.arange(max_length - length)])
  offset = make_offset(res_idx, Ls, cyclic_peptide)
  prev = {"prev_msa_first_row": np.zeros([max_length, 256]),
          "prev_pair":          np.zeros([max_length, max_length, 128]),
          "prev_pos":           np.zeros([max_length, 37, 3])}
  outs, positions, plddts, paes = [], [], [], []
  current_seq = (ori_sequence, cyclic_peptide)

# run only the recycles not already computed for this sequence
with tqdm.notebook.tqdm(total=recycles + 1, bar_format=TQDM_BAR_FORMAT) as pbar:
  pbar.update(min(len(outs), recycles + 1))
  while len(outs) < recycles + 1:
    O = fold_once(MODEL_PARAMS, INPUTS, aatype, length, res_idx, offset, prev)
    O = jax.tree_util.tree_map(np.asarray, O)
    prev = O["prev"]
    outs.append(O)
    positions.append(O["xyz"][:length])
    plddts.append(O["plddt"][:length] * 100)
    paes.append(O["pae"][:length, :length])
    pbar.update(1)

print(f"length={length}  recycle={recycles}  mean pLDDT={plddts[recycles].mean():.1f}"
      + ("  [cyclic]" if cyclic_peptide else ""))
ss_found = save_pdb(outs[recycles], aatype, Ls, "out.pdb", cyclic=cyclic_peptide)
if ss_found:
  print(f"{len(ss_found)} disulfide bond(s): " +
        ", ".join(f"{c1}{r1}-{c2}{r2} ({d:.2f} Å)" for c1, r1, c2, r2, d in ss_found) +
        "  [AlphaFold often draws these too short; energy minimization in part 2 relaxes them to ~2.05 Å]")
if cyclic_peptide:
  ends = np.linalg.norm(positions[recycles][0, 1] - positions[recycles][Ls[0] - 1, 1])
  print(f"CA(1)-CA({Ls[0]}) distance: {ends:.1f} Å  (about 3.8 Å in a closed macrocycle)")
view = show_pdb(open("out.pdb").read(), show_sidechains, show_mainchains,
                {"confidence": "pLDDT"}.get(color, color),
                Ls=Ls, color_HP=True, size=(800, 480))
view.setHoverable({}, True,
  """function(atom,viewer,event,container){if(!atom.label){atom.label=viewer.addLabel("      "+atom.resn+":"+atom.resi,{position:atom,backgroundColor:'mintcream',fontColor:'black'});}}""",
  """function(atom,viewer){if(atom.label){viewer.removeLabel(atom.label);delete atom.label;}}""")
view.show()
if color == "confidence":
  print("cartoon colored by pLDDT: red = low (<50) ... blue = high (>90)")
plot_confidence(plddts[recycles], paes[recycles], Ls=Ls, style=PAE_colors)
plt.show()


In [ ]:
#@title Animate
#@markdown - Shows how the structure changes across recycles (set `recycles` > 0 above first).
if len(positions) < 2:
  print("Set recycles > 0 in the cell above and run it again, then run this cell.")
else:
  display(HTML(make_animation(positions[:recycles + 1], np.asarray(plddts[:recycles + 1]), Ls=Ls)))


In [ ]:
#@title Download the predicted structure
#@markdown Saves the current prediction as a PDB file named after the sequence. Use it as the starting structure for the molecular dynamics exercise (Part 2).
import shutil
pdb_name = re.sub(r"[^A-Za-z0-9]", "", ori_sequence)[:20] + ("_cyclic" if cyclic_peptide else "") + ".pdb"
shutil.copy("out.pdb", pdb_name)
try:
  from google.colab import files
  files.download(pdb_name)
except ImportError:
  print("saved", pdb_name)
